# Templisafe - query parameterization use case

In [3]:
from typing import Final

## Example 1

In this section we will demonstrate the usage of the main features of the library with a simple example.

### Supported configuration types

You can access all the supported configuration types through the `ContentType` **enum**.

In [4]:
from templisafe import ContentType

ContentType._member_names_, ContentType._member_map_

(['TEXT', 'YAML', 'JSON', 'TOML', 'XML'],
 {'TEXT': <ContentType.TEXT: 'text'>,
  'YAML': <ContentType.YAML: 'yaml'>,
  'JSON': <ContentType.JSON: 'json'>,
  'TOML': <ContentType.TOML: 'toml'>,
  'XML': <ContentType.XML: 'xml'>})

The `TEXT` content type is used for template definitions. Schemas and variants may be defined using any supported configuration language: in this notebook, `YAML` is used.

### `Source` and `SourceSettings` objects

A `Source` represents an abstract input, which can be inline content, a local file or a remote cloud resource. All supported source types are listed in the `SourceKind` **enum**.

In [5]:
from templisafe import SourceKind

SourceKind._member_names_, ContentType._member_map_

(['INLINE',
  'LOCAL',
  'HTTP',
  'AWS_S3_BUCKET',
  'AWS_SECRETS_MANAGER',
  'AWS_SSM_PARAMETER',
  'AWS_DYNAMODB',
  'CUSTOM'],
 {'TEXT': <ContentType.TEXT: 'text'>,
  'YAML': <ContentType.YAML: 'yaml'>,
  'JSON': <ContentType.JSON: 'json'>,
  'TOML': <ContentType.TOML: 'toml'>,
  'XML': <ContentType.XML: 'xml'>})

You don’t need to create `Source` objects manually, the library handles them for you. Your only responsibility is to define a `Settings` object for the source, typically a `SourceSettings`, specifying the required configurations.

In the following, we show the different options you have to create a `Settings` object.

#### Creating settings using `SourceSettings.create`

You can easily create a `SourceSettings` using the `SourceSettings.create` method, providing the `SourceKind` and any necessary configuration parameters.

In [6]:
from templisafe import SourceSettings

inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline",                  # Enum is automatically parsed 
    content_type="text",            # Enum is automatically parsed 
    content="Hello {{ name }}!", 
)
inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content='Hello {{ name }}!')

#### Creating setting using `SourceSettings.from_<language>`

Alternatively, you can create a `SourceSettings` directly from a configuration string using the `SourceSettings.from_<language>` method.

For example, for a `YAML` configuration you will use the `SourceSettings.from_yaml` method.

In [7]:
json_str: str = '{ "schema": { "var1": "int", "var2": "list[str]" } }'

inline_yaml_settings: str = f"""
kind: inline
content_type: json
content: '{json_str}'
"""

inline_source_settings: SourceSettings = SourceSettings.from_yaml(inline_yaml_settings)
inline_source_settings

InlineSourceSettings(content_type=<ContentType.JSON: 'json'>, content='{ "schema": { "var1": "int", "var2": "list[str]" } }')

Another example: for a `JSON` configuration use `SourceSettings.from_json`.

In [8]:
local_json_settings: str = """
{
    "kind": "local",
    "path": "/tmp/query.sql.j2"
}
"""

local_source_settings: SourceSettings = SourceSettings.from_json(local_json_settings)
local_source_settings

LocalSourceSettings(content_type=None, path='/tmp/query.sql.j2')

### Resources definition

Now that you know how to define `SourceSettings`, let’s proceed to define all the resource configurations required for this use case.

#### Template

Define the **template** using an inline source.

In [9]:
sql_template_content: str = """SELECT
{% for column in columns %}  {{ column }}{% if not loop.last %},{% endif %}
{% endfor %}FROM orders
WHERE TRUE
  AND status = '{{ status }}'
  AND total_amount >= {{ min_total }}
  AND created_at >= '{{ start_date }}'
  AND created_at < '{{ end_date }}'
{% if customer_ids %}  AND customer_id IN (
{% for customer_id in customer_ids %}    {{ customer_id }}{% if not loop.last %},{% endif %}
{% endfor %}  )
{% endif %}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content=sql_template_content, 
    content_type="text"           # Use text for template contents
)
template_inline_source_settings


InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT\n{% for column in columns %}  {{ column }}{% if not loop.last %},{% endif %}\n{% endfor %}FROM orders\nWHERE TRUE\n  AND status = '{{ status }}'\n  AND total_amount >= {{ min_total }}\n  AND created_at >= '{{ start_date }}'\n  AND created_at < '{{ end_date }}'\n{% if customer_ids %}  AND customer_id IN (\n{% for customer_id in customer_ids %}    {{ customer_id }}{% if not loop.last %},{% endif %}\n{% endfor %}  )\n{% endif %}\n")

#### Schema

Define the **schema** using an inline source.

In [10]:
schema_content: str = """
schema:
  columns:
    type: list[str]
    default:
      - order_id
      - customer_id
      - status
      - total_amount
      - created_at
    metadata:
      description: Columns to project in the generated report query
      title: SELECT columns

  status:
    type: str
    default: PAID
    constraints:
      max_length: 20

  min_total:
    type: float
    default: 0.0
    constraints:
      ge: 0

  start_date: str

  end_date: str

  customer_ids:
    type: list[int]
    default: []
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content_type=ContentType.YAML, 
    content=schema_content
 )
schema_inline_source_settings


InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='\nschema:\n  columns:\n    type: list[str]\n    default:\n      - order_id\n      - customer_id\n      - status\n      - total_amount\n      - created_at\n    metadata:\n      description: Columns to project in the generated report query\n      title: SELECT columns\n\n  status:\n    type: str\n    default: PAID\n    constraints:\n      max_length: 20\n\n  min_total:\n    type: float\n    default: 0.0\n    constraints:\n      ge: 0\n\n  start_date: str\n\n  end_date: str\n\n  customer_ids:\n    type: list[int]\n    default: []\n')

#### Variants

Define a *single unnamed* **variant** using an inline source.

In [11]:
variants_content: str = """
variants:
  columns:
    - order_id
    - customer_id
    - total_amount
    - created_at

  # status -> defaulted
  min_total: 25.0
  start_date: "2024-01-01"
  end_date: "2024-02-01"
  customer_ids:
    - 101
    - 102
    - 103
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline", 
    content_type="yaml", 
    content=variants_content
)
variants_inline_source_settings


InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='\nvariants:\n  columns:\n    - order_id\n    - customer_id\n    - total_amount\n    - created_at\n\n  # status -> defaulted\n  min_total: 25.0\n  start_date: "2024-01-01"\n  end_date: "2024-02-01"\n  customer_ids:\n    - 101\n    - 102\n    - 103\n')

### Compilation

We can now compile the template against the schema to verify that all variables are correctly defined.

#### Create the `Templater`

Create the `Templater` object using default configurations.

In [12]:
from templisafe import Templater, TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()

templater: Templater = factory.create()
templater

#### Compile

Compile the template with the given schema.

In [13]:
from templisafe import Compilation

compilation: Compilation = templater.compile(
    template=template_inline_source_settings,
    schema=schema_inline_source_settings
)

compilation.outcome

<Outcome.SUCCESS: 0>

Inspect the `Compilation` object.

In [14]:
compilation

Compilation(outcome=<Outcome.SUCCESS: 0>, message='Query successfully compiled with schema', diagnostics=(), _spec=CompilationSpec(template=Template(template_str="SELECT\n{% for column in columns %}  {{ column }}{% if not loop.last %},{% endif %}\n{% endfor %}FROM orders\nWHERE TRUE\n  AND status = '{{ status }}'\n  AND total_amount >= {{ min_total }}\n  AND created_at >= '{{ start_date }}'\n  AND created_at < '{{ end_date }}'\n{% if customer_ids %}  AND customer_id IN (\n{% for customer_id in customer_ids %}    {{ customer_id }}{% if not loop.last %},{% endif %}\n{% endfor %}  )\n{% endif %}\n", vars={'customer_ids', 'columns', 'status', 'min_total', 'end_date', 'start_date'}), schema=Schema(model_cls=<class 'abc.ModelSchema'>)))

The schema generated is nothing but a **pydantic model**. 

In [15]:
compilation.compiled.schema.model_cls

abc.ModelSchema

In [16]:
compilation.compiled.schema.model_cls.model_fields

{'columns': FieldInfo(annotation=list[str], required=False, default=['order_id', 'customer_id', 'status', 'total_amount', 'created_at'], title='SELECT columns', description='Columns to project in the generated report query', json_schema_extra={'_index': 0}),
 'status': FieldInfo(annotation=str, required=False, default='PAID', json_schema_extra={'_index': 1}, metadata=[MaxLen(max_length=20)]),
 'min_total': FieldInfo(annotation=float, required=False, default=0.0, json_schema_extra={'_index': 2}, metadata=[Ge(ge=0)]),
 'start_date': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 3}),
 'end_date': FieldInfo(annotation=str, required=True, json_schema_extra={'_index': 4}),
 'customer_ids': FieldInfo(annotation=list[int], required=False, default=[], json_schema_extra={'_index': 5})}

#### Compile without a `Schema`

You can also decide to provide no schema when compiling a template: in this case, all variables will be parsed as `object` with default to `None`.

In [17]:
compilation_empty_schema: Compilation = templater.compile(template=template_inline_source_settings)      # No schema provided
compilation_empty_schema.outcome

<Outcome.SUCCESS: 0>

In [18]:
compilation_empty_schema.message

'Query successfully compiled with empty schema'

In [19]:
from templisafe import CompilationSpec, Schema

compiled_empty_schema: CompilationSpec = compilation_empty_schema.compiled
empty_schema: Schema = compiled_empty_schema.schema
empty_schema.model_cls.model_fields

{'customer_ids': FieldInfo(annotation=object, required=False, default=None),
 'columns': FieldInfo(annotation=object, required=False, default=None),
 'status': FieldInfo(annotation=object, required=False, default=None),
 'min_total': FieldInfo(annotation=object, required=False, default=None),
 'end_date': FieldInfo(annotation=object, required=False, default=None),
 'start_date': FieldInfo(annotation=object, required=False, default=None)}

### Rendering

After the compilation, we can proceed with the template rendering using the defined **variants**.

In [20]:
from templisafe import InlineSourceSettings

assert isinstance(variants_inline_source_settings, InlineSourceSettings)
print(variants_inline_source_settings.content)


variants:
  columns:
    - order_id
    - customer_id
    - total_amount
    - created_at

  # status -> defaulted
  min_total: 25.0
  start_date: "2024-01-01"
  end_date: "2024-02-01"
  customer_ids:
    - 101
    - 102
    - 103



Render the template with the given variants.

In [21]:
from templisafe import Rendering

rendering: Rendering = templater.render(
    compiled=compilation.compiled,
    variants=variants_inline_source_settings
)

rendering.outcome

<Outcome.SUCCESS: 0>

Inspect the `Rendering` object.

In [22]:
rendering

Rendering(outcome=<Outcome.SUCCESS: 0>, message='Rendering successful', diagnostics=(), _spec=RenderingSpec(_param_by_name={'default_1': Parameterization(variant=Variant(name='default_1', _binding_by_name={'columns': Binding(index=0, name='columns', value=['order_id', 'customer_id', 'total_amount', 'created_at']), 'min_total': Binding(index=1, name='min_total', value=25.0), 'start_date': Binding(index=2, name='start_date', value='2024-01-01'), 'end_date': Binding(index=3, name='end_date', value='2024-02-01'), 'customer_ids': Binding(index=4, name='customer_ids', value=[101, 102, 103])}), rendered_str="SELECT\n  order_id,\n  customer_id,\n  total_amount,\n  created_at\nFROM orders\nWHERE TRUE\n  AND status = 'PAID'\n  AND total_amount >= 25.0\n  AND created_at >= '2024-01-01'\n  AND created_at < '2024-02-01'\n  AND customer_id IN (\n    101,\n    102,\n    103\n  )\n")}))

The variant had no name associated, so a default name was generated.

In [23]:
from templisafe import RenderingSpec

rendered: RenderingSpec = rendering.rendered 
rendered.names

{'default_1'}

Rendered template.

In [24]:
for r in rendered.parameterizations: 
    print(r.rendered_str)

SELECT
  order_id,
  customer_id,
  total_amount,
  created_at
FROM orders
WHERE TRUE
  AND status = 'PAID'
  AND total_amount >= 25.0
  AND created_at >= '2024-01-01'
  AND created_at < '2024-02-01'
  AND customer_id IN (
    101,
    102,
    103
  )



Rendered template bindings.

In [25]:
rendered.parameterizations[0].variant.names

{'columns', 'customer_ids', 'end_date', 'min_total', 'start_date'}

In [26]:
rendered.parameterizations[0].variant.bindings

[Binding(index=0, name='columns', value=['order_id', 'customer_id', 'total_amount', 'created_at']),
 Binding(index=1, name='min_total', value=25.0),
 Binding(index=2, name='start_date', value='2024-01-01'),
 Binding(index=3, name='end_date', value='2024-02-01'),
 Binding(index=4, name='customer_ids', value=[101, 102, 103])]

## Example 2

In this section we will show a more realistic example, where the resources (**template**, **schema** and **variants**) are defined through configuration files.

### Configuration files

In [27]:
BASE_PATH: Final[str] = "./resources"

Template definition.

In [28]:
TEMPLATE_PATH: Final[str] = BASE_PATH + "/template.sql.j2"
MACROS_PATH: Final[str] = BASE_PATH + "/macros"

with open(TEMPLATE_PATH) as f:
    print(f.read())


{% import "order_filters.sql.j2" as order_filters -%}
SELECT
{% for column in columns %}  {{ column }}{% if not loop.last %},{% endif %}
{% endfor %}FROM orders
WHERE TRUE
  AND status = '{{ status }}'
  AND total_amount >= {{ min_total }}
  AND created_at >= '{{ start_date }}'
  AND created_at < '{{ end_date }}'
{{ order_filters.customer_id_filter(customer_ids) }}



Schema definition.

In [29]:
SCHEMA_PATH: Final[str] = BASE_PATH + "/schema.yaml"
with open(SCHEMA_PATH) as f:
    print(f.read())


schema:
  columns:
    type: list[str]
    default:
      - order_id
      - customer_id
      - status
      - total_amount
      - created_at
    metadata:
      description: Columns to project in the generated report query
      title: SELECT columns

  status:
    type: str
    default: PAID
    constraints:
      max_length: 20

  min_total:
    type: float
    default: 0.0
    constraints:
      ge: 0

  start_date: str

  end_date: str

  customer_ids:
    type: list[int]
    default: []



Variants definition.

In [30]:
VARIANTS_PATH_1: Final[str] = BASE_PATH + "/variants1.yaml"
with open(VARIANTS_PATH_1) as f:
    print(f.read())

variants:
  - name: recent_paid_orders
    bindings:
      columns:
        - order_id
        - customer_id
        - total_amount
        - created_at

      # status -> defaulted
      min_total: 25.0
      start_date: "2024-01-01"
      end_date: "2024-02-01"
      customer_ids:
        - 101
        - 102
        - 103

  - name: high_value_orders
    bindings:
      columns:
        - order_id
        - customer_id
        - status
        - total_amount
        - created_at

      status: PAID
      min_total: 500.0
      start_date: "2024-01-01"
      end_date: "2024-04-01"
      customer_ids:
        - 204
        - 305



In [31]:
VARIANTS_PATH_2: Final[str] = BASE_PATH + "/variants2.yaml"
with open(VARIANTS_PATH_2) as f:
    print(f.read())

variants:
  all_paid_orders:
    columns:
      - order_id
      - customer_id
      - total_amount
      - created_at

    # status -> defaulted
    # min_total -> defaulted
    start_date: "2024-01-01"
    end_date: "2024-02-01"
    # customer_ids -> defaulted



### Sources

Use a `LocalSource` to load configurations from a file. 

When the file uses a standard extension, the content type is inferred automatically.

In [32]:
template_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=TEMPLATE_PATH        # No content_type specified
)

schema_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=SCHEMA_PATH          # No content_type specified
)

variants1_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=VARIANTS_PATH_1      # No content_type specified
)

variants2_local_source_settings: SourceSettings = SourceSettings.create(
    kind="local", path=VARIANTS_PATH_2      # No content_type specified
)

template_local_source_settings, schema_local_source_settings, variants1_local_source_settings, variants2_local_source_settings

(LocalSourceSettings(content_type=None, path='./resources/template.sql.j2'),
 LocalSourceSettings(content_type=None, path='./resources/schema.yaml'),
 LocalSourceSettings(content_type=None, path='./resources/variants1.yaml'),
 LocalSourceSettings(content_type=None, path='./resources/variants2.yaml'))

### Template engine settings

The file template imports a Jinja macro, so the Jinja environment receives a file-system loader pointing to the macro directory.


In [33]:
from jinja2 import FileSystemLoader
from templisafe import TemplateEngineSettings

template_engine_settings: TemplateEngineSettings = TemplateEngineSettings.create(
    engine_kind="jinja",
    config={"loader": FileSystemLoader(MACROS_PATH)},
)
template_engine_settings


TemplateEngineSettings(engine_kind=<TemplateEngineKind.JINJA: 'jinja'>, config={'loader': <jinja2.loaders.FileSystemLoader object at 0x0000014EFB7F70E0>})

### Build - compilation and rendering in one step

Use `build` to `compile` and `render` in one step.

In [34]:
from templisafe import Build

build: Build = templater.build(
    template=template_local_source_settings,
    schema=schema_local_source_settings,
    variants=[variants1_local_source_settings, variants2_local_source_settings],
    template_engine=template_engine_settings,
)

build.outcome


<Outcome.SUCCESS: 0>

The `Build` result contains both the `Compilation` and `Rendering` objects.

Inspect the `Compilation`.

In [35]:
compilation: Compilation = build.compilation
compilation.outcome, compilation.message

(<Outcome.SUCCESS: 0>, 'Query successfully compiled with schema')

Inspect the `Rendering`.

In [36]:
rendering: Rendering = build.rendering
rendering.outcome, rendering.message

(<Outcome.SUCCESS: 0>, 'Rendering successful')

In [37]:
rendering.rendered.names

{'all_paid_orders', 'high_value_orders', 'recent_paid_orders'}

In [38]:
from templisafe import Parameterization, Variant

parameterizations: list[Parameterization] = rendering.rendered.parameterizations
for par in parameterizations:
    variant: Variant = par.variant
    print("-" * 50)
    print(f"Variant '{variant.name}':")
    print("-" * 50)
    for b in variant.bindings:
        print(b)

--------------------------------------------------
Variant 'recent_paid_orders':
--------------------------------------------------
Binding(index=0, name='columns', value=['order_id', 'customer_id', 'total_amount', 'created_at'])
Binding(index=1, name='min_total', value=25.0)
Binding(index=2, name='start_date', value='2024-01-01')
Binding(index=3, name='end_date', value='2024-02-01')
Binding(index=4, name='customer_ids', value=[101, 102, 103])
--------------------------------------------------
Variant 'high_value_orders':
--------------------------------------------------
Binding(index=0, name='columns', value=['order_id', 'customer_id', 'status', 'total_amount', 'created_at'])
Binding(index=1, name='status', value='PAID')
Binding(index=2, name='min_total', value=500.0)
Binding(index=3, name='start_date', value='2024-01-01')
Binding(index=4, name='end_date', value='2024-04-01')
Binding(index=5, name='customer_ids', value=[204, 305])
--------------------------------------------------
Va

Note that for the variant '**all_paid_orders**' no bindings for '*status*', '*min_total*' and '*customer_ids*' were specified: the variables defaulted to the values defined in the schema.

In [39]:
for variant_name, variant in rendering.rendered.mapping.items(): 
    print("-" * 50)
    print(f"Variant '{variant_name}':")
    print("-" * 50)
    print(variant.rendered_str)

--------------------------------------------------
Variant 'recent_paid_orders':
--------------------------------------------------
SELECT
  order_id,
  customer_id,
  total_amount,
  created_at
FROM orders
WHERE TRUE
  AND status = 'PAID'
  AND total_amount >= 25.0
  AND created_at >= '2024-01-01'
  AND created_at < '2024-02-01'
  AND customer_id IN (
    101,
    102,
    103
  )

--------------------------------------------------
Variant 'high_value_orders':
--------------------------------------------------
SELECT
  order_id,
  customer_id,
  status,
  total_amount,
  created_at
FROM orders
WHERE TRUE
  AND status = 'PAID'
  AND total_amount >= 500.0
  AND created_at >= '2024-01-01'
  AND created_at < '2024-04-01'
  AND customer_id IN (
    204,
    305
  )

--------------------------------------------------
Variant 'all_paid_orders':
--------------------------------------------------
SELECT
  order_id,
  customer_id,
  total_amount,
  created_at
FROM orders
WHERE TRUE
  AND status

<!-- ## Diagnostics examples -->

## Errors validation examples

In this section, we'll show how `templisafe` can safely prevent common template definition errors, both in the **compilation** and **rendering** steps. 

### Diagnostic policy

You can access all the supported configuration types through the `ContentType` **enum**.

In [40]:
from templisafe import ContentType

ContentType._member_names_, ContentType._member_map_

(['TEXT', 'YAML', 'JSON', 'TOML', 'XML'],
 {'TEXT': <ContentType.TEXT: 'text'>,
  'YAML': <ContentType.YAML: 'yaml'>,
  'JSON': <ContentType.JSON: 'json'>,
  'TOML': <ContentType.TOML: 'toml'>,
  'XML': <ContentType.XML: 'xml'>})

When instantiating a `Templater`, you can define its behavior for handling warnings and errors by specifying a `DiagnosticPolicy` from the corresponding enum.

In [41]:
from templisafe import DiagnosticPolicy

DiagnosticPolicy._member_names_, DiagnosticPolicy._member_map_

(['IGNORE', 'LOG', 'STRICT'],
 {'IGNORE': <DiagnosticPolicy.IGNORE: 'ignore'>,
  'LOG': <DiagnosticPolicy.LOG: 'log'>,
  'STRICT': <DiagnosticPolicy.STRICT: 'strict'>})

### Compilation

#### Undeclared variables

An **undeclared variable** is a variable defined in the template but not in the schema, which causes the compilation to fail.

Template

In [42]:
# The variable 'undeclared' is used in the template but will not be defined in the schema
sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
  AND m.code = '{{ undeclared }}'     
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="text",
    content=sql_template_content 
)
template_inline_source_settings

InlineSourceSettings(content_type=<ContentType.TEXT: 'text'>, content="SELECT \n  {{ col1 }}, {{ col2 }}\nFROM users u\nWHERE TRUE\n  AND u.status = '{{ user_status }}'\n  AND u.age > {{ user_age_lower }}\n  AND m.code = '{{ undeclared }}'     \n")

Schema

In [43]:
# No variable 'undeclared' in the schema
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=schema_content,
)

schema_inline_source_settings

InlineSourceSettings(content_type=<ContentType.YAML: 'yaml'>, content='schema:\n  col1: str\n  col2: str\n  user_status: str\n  user_age_lower: int \n')

In [46]:
from templisafe import Templater, TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="ignore")        # Use ignore policy to avoid raising errors

compilation: Compilation = templater.compile(
    template=template_inline_source_settings,
    schema=schema_inline_source_settings
)

compilation.outcome

CompilationFailureError: Template compilation failed with outcome ERROR: Query compilation failed
Diagnostics:
[ERROR] variable=undeclared: Undeclared variable: 'undeclared'

The **compilation** failed. Inspecting the `Compilation` object we can see the error messages.

In [45]:
compilation.message, compilation.diagnostics

('Query successfully compiled with schema', ())

#### Unused variables

An **unused variable** is a variable defined in the schema but not in the template, which causes a warning in the compilation.

Template

In [ ]:
# No variable 'unused' in the template
sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="text",
    content=sql_template_content
    )
template_inline_source_settings

Schema

In [ ]:
# Variable 'unused' defined in the schema but not used in the template
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
  unused: any
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=schema_content,
)

schema_inline_source_settings

In [ ]:
from templisafe import Templater, TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="log")       # Use log policy to log warnings and raise errors

compilation: Compilation = templater.compile(
    template=template_inline_source_settings,
    schema=schema_inline_source_settings
)

compilation.outcome

The **compilation** was completed with warnings. Inspecting the `Compilation` object we can see the warning messages.

In [ ]:
compilation.message, compilation.diagnostics

### Rendering

In the rendering process we start from an already compiled template, consisting in a `Compilation` object.

Template

In [ ]:
sql_template_content: str = """SELECT 
  {{ col1 }}, {{ col2 }}
FROM users u
WHERE TRUE
  AND u.status = '{{ user_status }}'
  AND u.age > {{ user_age_lower }}
"""

template_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="text",
    content=sql_template_content 
)
template_inline_source_settings

Schema

In [ ]:
schema_content: str = """schema:
  col1: str
  col2: str
  user_status: str
  user_age_lower: int 
"""

schema_inline_source_settings: SourceSettings = SourceSettings.create(
    kind="inline",
    content_type="yaml",
    content=schema_content
)

schema_inline_source_settings

In [ ]:
from templisafe import Templater, TemplaterFactory

factory: TemplaterFactory = TemplaterFactory()
templater: Templater = factory.create(diagnostic_policy="ignore")        # Use ignore policy to avoid raising errors

compilation: Compilation = templater.compile(
    template=template_inline_source_settings,
    schema=schema_inline_source_settings
)

compilation.outcome

In [ ]:
from templisafe import CompilationSpec

compiled: CompilationSpec = compilation.compiled
compiled

In [ ]:
compiled.schema.model_cls.model_fields

#### Undeclared binding

An **undeclared binding** consists in a variable (without a default) defined in the schema without a corresponding binding in a variant, which causes the rendering to fail.

In [ ]:
# The variant is missing the binding 'user_age_lower', 
# which was defined in the schema as a variable without a default
variants_content: str = """variants:
  col1: id
  col2: name
  user_status: ACTIVE
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=variants_content
    )
variants_inline_source_settings

In [ ]:
from templisafe import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants=variants_inline_source_settings
)

rendering.outcome

The **rendering** failed. Inspecting the `Rendering` object we can see the error messages.

In [ ]:
rendering.message, rendering.diagnostics

#### Unused binding

An **unused binding** is a binding defined in a variant without a corresponding variable in the schema, which causes a warning in the rendering.

In [ ]:
# Binding 'unused' was not defined in the schema
variants_content: str = """variants:
  col1: id
  col2: name
  user_status: ACTIVE
  user_age_lower: 18
  unused: unused 
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=variants_content
    )
variants_inline_source_settings

In [ ]:
from templisafe import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants=variants_inline_source_settings
)

rendering.outcome

The **rendering** was completed with warnings. Inspecting the `Rendering` object we can see the warning messages.

In [ ]:
rendering.message, rendering.diagnostics

#### Wrong typed binding

A **wrong typed binding** is a binding defined in the variant with a *type* different from the one defined in the corresponding schema variable, which causes the rendering to fail.

In [ ]:
variants_content: str = """
variants:
  col1: 5.67                      # Should be a string
  col2: name
  user_status: 1                  # Should be a string
  user_age_lower: [1, 2, 3]       # Should be an int
"""

variants_inline_source_settings: SourceSettings = SourceSettings.create(
    kind='inline',
    content_type="yaml",
    content=variants_content
    )
variants_inline_source_settings

In [ ]:
from templisafe import Rendering

rendering: Rendering = templater.render(
    compiled=compiled,
    variants=variants_inline_source_settings
)

rendering.outcome

The **rendering** failed. Inspecting the `Rendering` object we can see the error messages.

In [ ]:
rendering.message, rendering.diagnostics